# 05. Doubly Linked List and Design Problems

A **doubly** linked list has `prev` and `next`. You can delete a node in **O(1)** if you hold the node, because you can reach its neighbours. This is the engine behind LRU Cache and browser history.

**Learning order:** DOUBLY LINKED NODE → LRU CACHE → BROWSER HISTORY → COPY WITH RANDOM POINTER → INTERVIEW THINKING

## 1. Why doubly?

- Singly linked: to delete a node you need its **previous** node (O(n) to find).
- Doubly: `node.prev.next = node.next; node.next.prev = node.prev` → O(1).
- Costs more memory (extra pointer).

**Sentinel nodes:** use a fake `head` and `tail` so inserting/removing never has `None` cases.

## 2. LRU Cache (Least Recently Used)

**Requirement:** `get(key)` and `put(key, value)` in **O(1)**. When the cache is full, evict the **least recently used** key.

**Design:** a **hash map** (`key → node`) for O(1) lookup + a **doubly linked list** ordered by recency (front = most recent). Every use moves the node to the front. Eviction removes the node before the tail sentinel.

Real life: your phone's "recent apps" list. Real systems: caches in browsers, databases, CDNs.

In [1]:
class Node:
    def __init__(self, key=0, val=0):
        self.key, self.val = key, val
        self.prev = self.next = None

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.cache = {}                                   # key -> node
        self.head, self.tail = Node(), Node()             # sentinels
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, node):
        node.prev.next, node.next.prev = node.next, node.prev

    def _add_front(self, node):
        node.next, node.prev = self.head.next, self.head
        self.head.next.prev = node
        self.head.next = node

    def get(self, key):
        if key not in self.cache:
            return -1
        node = self.cache[key]
        self._remove(node)
        self._add_front(node)                             # now most recent
        return node.val

    def put(self, key, value):
        if key in self.cache:
            self._remove(self.cache[key])
        node = Node(key, value)
        self.cache[key] = node
        self._add_front(node)
        if len(self.cache) > self.cap:
            lru = self.tail.prev                          # least recent
            self._remove(lru)
            del self.cache[lru.key]

c = LRUCache(2)
c.put(1, 1); c.put(2, 2)
print(c.get(1))      # 1
c.put(3, 3)          # evicts key 2
print(c.get(2))      # -1
c.put(4, 4)          # evicts key 1
print(c.get(1), c.get(3), c.get(4))   # -1 3 4

1
-1
-1 3 4


## 3. Browser history (back / forward)

Each visited page is a node in a doubly linked list. `back(steps)` moves left, `forward(steps)` moves right, `visit(url)` adds a new node and **discards the forward history**.

In [2]:
class BrowserHistory:
    def __init__(self, homepage):
        self.cur = Node(0, homepage)

    def visit(self, url):
        node = Node(0, url)
        self.cur.next = node
        node.prev = self.cur
        self.cur = node

    def back(self, steps):
        while steps and self.cur.prev:
            self.cur = self.cur.prev
            steps -= 1
        return self.cur.val

    def forward(self, steps):
        while steps and self.cur.next:
            self.cur = self.cur.next
            steps -= 1
        return self.cur.val

b = BrowserHistory("leetcode.com")
b.visit("google.com"); b.visit("facebook.com"); b.visit("youtube.com")
print(b.back(1), b.back(1), b.forward(1))   # facebook.com google.com facebook.com
b.visit("linkedin.com")
print(b.forward(2), b.back(2), b.back(7))   # linkedin.com google.com leetcode.com

facebook.com google.com facebook.com
linkedin.com google.com leetcode.com


## 4. Copy List with Random Pointer

Each node has `next` **and** `random` (can point anywhere). Make a **deep copy**.

**Hash map approach:** pass 1 creates a copy of every node (`old → new`). Pass 2 sets `new.next = map[old.next]`, `new.random = map[old.random]`. O(n) time, O(n) space.

(An O(1) space version interleaves copies in the original list, but the hash map is usually enough in interviews.)

In [3]:
class RNode:
    def __init__(self, val):
        self.val = val
        self.next = None
        self.random = None

def copy_random_list(head):
    if not head:
        return None
    old_to_new = {None: None}
    cur = head
    while cur:
        old_to_new[cur] = RNode(cur.val)
        cur = cur.next
    cur = head
    while cur:
        old_to_new[cur].next = old_to_new[cur.next]
        old_to_new[cur].random = old_to_new[cur.random]
        cur = cur.next
    return old_to_new[head]

a, b = RNode(1), RNode(2)
a.next = b; a.random = b; b.random = b
copy = copy_random_list(a)
print(copy is not a, copy.next.val, copy.random is copy.next, copy.next.random is copy.next)   # True 2 True True

True 2 True True


## 5. Interview thinking

**Signals:** "O(1) get and put", "evict least recently used", "back and forward", "delete a node given only that node".

**Say:** "I need O(1) lookup, so a hash map, and O(1) reorder/removal, so a doubly linked list. The map stores key → node."

**Common mistakes:** forgetting to store the **key** inside the node (needed to delete from the map on eviction), updating pointers in the wrong order, not moving a node to the front on `get`, not using sentinels, not removing the old node on `put` of an existing key.

**Python shortcut:** `collections.OrderedDict` with `move_to_end` and `popitem(last=False)` does the same in a few lines. Know both.

## Quick Revision

- Doubly linked: O(1) delete when you hold the node.
- LRU = hash map (key → node) + doubly linked list (recency order) + sentinels.
- `get`: move to front. `put`: add front, evict `tail.prev` if full.
- Random pointer copy: map old → new, two passes.
- Store `key` in node so eviction can delete from the map.